# Retrieval Process Demonstration

This Jupyter Notebook demonstrates the retrieval process used in the `main.py` application. It includes code for loading data, processing URLs, and querying the FAISS index.

In [ ]:
import os
import pickle
from langchain import OpenAI
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import UnstructuredURLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Define the file path for the FAISS index
file_path = 'faiss_store_openai.pkl'

# Function to load and process URLs
def load_and_process_urls(urls):
    loader = UnstructuredURLLoader(urls=urls)
    data = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        separators=['\n\n', '\n', '.', ','],
        chunk_size=1000
    )
    docs = text_splitter.split_documents(data)
    embeddings = OpenAIEmbeddings()
    vectorstore_openai = FAISS.from_documents(docs, embeddings)
    return vectorstore_openai

# Example usage
urls = ['https://example.com/article1', 'https://example.com/article2']  # Replace with actual URLs
vectorstore = load_and_process_urls(urls)

# Save the FAISS index to a pickle file
with open(file_path, 'wb') as f:
    pickle.dump(vectorstore, f)

# Function to query the FAISS index
def query_faiss_index(query):
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            vectorstore = pickle.load(f)
            llm = OpenAI(temperature=0.9, max_tokens=500)
            chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=vectorstore.as_retriever())
            result = chain({'question': query}, return_only_outputs=True)
            return result
    else:
        return 'FAISS index not found.'

# Example query
query = 'What is the main topic of the articles?'
result = query_faiss_index(query)
print(result)
